[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimyortega55-collab/motor-OCR/blob/main/entrenamiento/colab_finetuning_real.ipynb)

# Fine-tuning real de pix2tex en Colab

A diferencia de `colab_prueba_humo.ipynb` (300 formulas, 1 epoca, solo para confirmar que el bucle corre), este notebook:

1. Genera un dataset sintetico mas grande con `entrenamiento/generar_dataset_sintetico.py`.
2. Lo empaqueta al formato `.pkl` que espera `pix2tex`, usando el mismo tokenizer del checkpoint pre-entrenado.
3. Corre el fine-tuning de verdad, mas epocas, guardando los checkpoints directo en Google Drive (Colab borra el disco local al desconectarse).

**Corre primero la prueba de humo si no lo hiciste** -- confirma que el entorno funciona antes de invertir tiempo en un dataset grande.

Antes de empezar: **Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU**.

## 0. (Solo si vas a usar VS Code) Abrir un tunel SSH hacia esta VM

Igual que en la prueba de humo: corre esto una sola vez aqui en el navegador si vas a seguir desde VS Code con Remote-SSH. Si te quedas en el navegador de Colab, saltatela.

In [2]:
!pip install -q colab-ssh --upgrade
from colab_ssh import launch_ssh_cloudflared
from getpass import getpass

clave_temporal = getpass("Clave temporal para la sesion SSH (no la reutilices): ")
launch_ssh_cloudflared(password=clave_temporal)

## 1. Confirmar que hay GPU asignada

In [ ]:
!nvidia-smi

import torch

# Sin GPU el fine-tuning es inviable (dias en vez de horas), asi que se corta
# aca en vez de descubrirlo despues de generar el dataset.
assert torch.cuda.is_available(), (
    "No hay GPU asignada. Entorno de ejecucion -> Cambiar tipo de entorno de "
    "ejecucion -> GPU, y volve a correr desde esta celda."
)
gpu = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | {gpu.name} | {gpu.total_memory / 1024**3:.1f} GB")

## 2. Montar Google Drive

El disco de Colab se borra al desconectarse. Los checkpoints del fine-tuning (los `.pth`, chicos) se guardan directo en Drive para no perderlos. El dataset generado (miles de imagenes chicas) se queda en el disco local de Colab -- es rapido de regenerar y escribir miles de archivos chicos en Drive es lento.

In [4]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/motor-ocr-finetuning"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Checkpoints en:", DRIVE_DIR)

Mounted at /content/drive
Checkpoints en: /content/drive/MyDrive/motor-ocr-finetuning


## 3. Clonar el repo

In [ ]:
import os

REPO_URL = "https://github.com/rimyortega55-collab/motor-OCR.git"
REPO_DIR = "/content/motor-OCR"

# La ruta del clon tiene que ser siempre la misma: el .pkl del dataset guarda
# las rutas de las imagenes relativas al directorio desde el que se empaqueto,
# asi que el cache de dataset en Drive solo se puede reusar si el repo vive
# siempre en /content/motor-OCR.
if os.path.isdir(REPO_DIR):
    # Colab se desconecta seguido y esta celda se vuelve a correr; `git clone`
    # falla si el directorio ya existe, asi que se actualiza en su lugar.
    !git -C /content/motor-OCR pull --ff-only
else:
    !git clone {REPO_URL} /content/motor-OCR

%cd /content/motor-OCR
!git log --oneline -1

## 4. Instalar dependencias

`pix2tex` para el modelo/entrenamiento, y una distribucion LaTeX (para `pdflatex`) porque `generar_dataset_sintetico.py` renderiza cada formula a imagen antes de rasterizarla con PyMuPDF. La instalacion de LaTeX tarda varios minutos.

`imagesize` y `python-Levenshtein` los importa `pix2tex` (en `dataset/dataset.py` y en `eval.py`) pero solo los declara en su extra `[train]`, no en la instalacion base -- sin ellos el empaquetado del dataset y la evaluacion revientan con `ModuleNotFoundError`. Se instalan explicitamente en vez de usar `pix2tex[train]` porque ese extra arrastra `torchtext`, que en Colab fuerza un downgrade de `torch`; `entrenamiento/_compat.py` ya sustituye lo unico que pix2tex usa de `torchtext` (`bleu_score`).

In [7]:
!pip install -q pix2tex wandb python-Levenshtein pymupdf imagesize
!apt-get -qq update
!apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended texlive-latex-recommended
!which pdflatex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.0/427.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 80.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.2 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1

In [ ]:
import shutil

import torch

# `pip install pix2tex` puede arrastrar una version de torch distinta a la que
# Colab trae preinstalada y dejar la GPU sin usar. Se comprueba explicitamente
# en vez de descubrirlo cuando el entrenamiento va 10x mas lento de lo esperado.
print("torch:", torch.__version__, "| cuda disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Se perdio el acceso a la GPU al instalar dependencias. Entorno de ejecucion "
    "-> Reiniciar entorno de ejecucion, y volve a correr desde la celda del clon "
    "(no hace falta reinstalar: los paquetes sobreviven al reinicio)."
)
assert shutil.which("pdflatex"), (
    "pdflatex no quedo en el PATH: revisa la salida de apt-get de la celda anterior."
)
print("pdflatex:", shutil.which("pdflatex"))

## 5. Parametros

Son un punto de partida razonable, no valores probados como optimos -- ajustalos segun cuanto tiempo/GPU tengas y lo que veas en las primeras corridas. Si Colab se queda sin memoria de GPU, baja `BATCHSIZE`/`MICRO_BATCHSIZE` primero.

In [8]:
N_TRAIN = 3000
N_VAL = 300
PROFUNDIDAD_MAX = 3

EPOCHS = 20
BATCHSIZE = 10
MICRO_BATCHSIZE = 5
LR = 1e-4  # mas bajo que el 1e-3 de la prueba de humo: ya partimos de un checkpoint entrenado, no de cero

# config_validacion.yaml trae sample_freq: 5 (evaluar cada 5 pasos), razonable para una
# corrida de humo de 300 formulas pero no aca: con 3000 formulas y batchsize 10 son 300
# pasos por epoca, o sea ~60 evaluaciones por epoca, y cada mejora guarda un checkpoint
# de ~100 MB en Drive. Se evalua ~una vez por epoca.
SAMPLE_FREQ = 300
SAVE_FREQ = 2  # guardar cada 2 epocas: 20 checkpoints de 100 MB llenarian Drive

## 6. Dataset: reusar el de Drive, o generarlo

Generar el dataset es lo que mas tarda de todo el notebook (`pdflatex` renderiza
una formula por vez, con reintentos) y el disco local de Colab se borra al
desconectarse. Por eso el dataset se cachea en Drive como **un solo `.tar`**:
escribir miles de PNG sueltos en Drive es lentisimo, un archivo grande no.

La primera corrida lo genera y lo cachea; las siguientes -incluida una
reanudacion despues de que Colab te desconecte- lo restauran en segundos.

In [ ]:
import os

DATASET_DIR = "entrenamiento/dataset_real"
DATASET_TAR = f"{DRIVE_DIR}/dataset_real.tar"

DATASET_EN_CACHE = os.path.exists(DATASET_TAR)
if DATASET_EN_CACHE:
    print(f"Restaurando el dataset cacheado desde {DATASET_TAR} ...")
    !tar -xf {DATASET_TAR} -C entrenamiento
    print("Listo: se saltan la generacion y el empaquetado.")
else:
    print("No hay dataset cacheado en Drive: se genera desde cero en las celdas siguientes.")
    print("Si cambiaste N_TRAIN / N_VAL / PROFUNDIDAD_MAX, borra el .tar de Drive "
          "para forzar la regeneracion.")

In [ ]:
if not DATASET_EN_CACHE:
    !python entrenamiento/generar_dataset_sintetico.py --n-train {N_TRAIN} --n-val {N_VAL} --profundidad-max {PROFUNDIDAD_MAX} --out {DATASET_DIR}

### 6.b Comprobar que el dataset sirve

`generar_dataset_sintetico.py` no falla cuando `pdflatex` no puede renderizar una
formula: deja la linea vacia en `formulas.txt` y sigue. Si LaTeX quedo mal
instalado, el resultado es un dataset vacio y el entrenamiento arrancaria contra
la nada. Se comprueba antes de invertir horas de GPU.

In [ ]:
from pathlib import Path

for split in ("train", "val"):
    dir_split = Path(DATASET_DIR) / split
    lineas = (dir_split / "formulas.txt").read_text(encoding="utf-8").splitlines()
    renderizadas = sum(1 for linea in lineas if linea.strip())
    imagenes = len(list((dir_split / "imagenes").glob("*.png")))
    print(f"{split}: {renderizadas}/{len(lineas)} formulas renderizadas, {imagenes} imagenes")
    assert renderizadas > 0, (
        f"El split '{split}' quedo vacio: pdflatex no renderizo ni una formula. "
        "Revisa la salida de la celda de instalacion de LaTeX."
    )
    if renderizadas < 0.8 * len(lineas):
        print(f"  AVISO: fallo mas del 20% de '{split}'; el dataset efectivo es "
              "mas chico que el pedido.")

## 7. Checkpoint pre-entrenado y empaquetado en `.pkl`

Baja los pesos pre-entrenados de pix2tex y, si el dataset no vino del cache, lo
empaqueta al formato `.pkl` que espera `pix2tex` y guarda el `.tar` en Drive.

El empaquetado usa **el mismo tokenizer que ya usa el checkpoint
pre-entrenado**: si se generara un tokenizer nuevo, el vocabulario no
coincidiria con los pesos pre-entrenados y el fine-tuning arrancaria de
embeddings sin sentido.

Ojo: el wheel de `pix2tex` **no** trae `weights.pth` (son ~100 MB, no estan en
el paquete de PyPI); solo trae el `tokenizer.json`. Los pesos se bajan aparte
con `download_checkpoints()`, que es lo que hace la celda si no los encuentra.

In [ ]:
import os

import pix2tex
from pix2tex.model.checkpoints.get_latest_checkpoint import download_checkpoints

base_pix2tex = os.path.dirname(pix2tex.__file__)
TOKENIZER = os.path.join(base_pix2tex, "model", "dataset", "tokenizer.json")
CHECKPOINT = os.path.join(base_pix2tex, "model", "checkpoints", "weights.pth")

if not os.path.exists(CHECKPOINT):
    download_checkpoints()  # baja weights.pth + image_resizer.pth desde el release de LaTeX-OCR

assert os.path.exists(TOKENIZER), f"No se encontro el tokenizer en {TOKENIZER}"
assert os.path.exists(CHECKPOINT), f"No se pudo descargar el checkpoint pre-entrenado a {CHECKPOINT}"
print("checkpoint:", CHECKPOINT, os.path.getsize(CHECKPOINT) // 1024**2, "MB")

if not DATASET_EN_CACHE:
    # Los .pkl guardan las rutas de las imagenes tal como se pasan aca, o sea
    # relativas a /content/motor-OCR. Por eso el clon siempre va a esa ruta fija.
    !python -m pix2tex.dataset.dataset -i {DATASET_DIR}/train/imagenes -e {DATASET_DIR}/train/formulas.txt -t {TOKENIZER} -o {DATASET_DIR}/train.pkl
    !python -m pix2tex.dataset.dataset -i {DATASET_DIR}/val/imagenes -e {DATASET_DIR}/val/formulas.txt -t {TOKENIZER} -o {DATASET_DIR}/val.pkl

    print("Cacheando el dataset en Drive para no tener que regenerarlo...")
    !tar -cf {DATASET_TAR} -C entrenamiento dataset_real
    print("Cacheado en", DATASET_TAR, os.path.getsize(DATASET_TAR) // 1024**2, "MB")
else:
    print("Dataset restaurado del cache: ya trae los .pkl empaquetados.")

## 8. Armar la config de fine-tuning

Parte de `config_validacion.yaml` (misma arquitectura que el checkpoint pre-entrenado) y sobreescribe lo que cambia para una corrida real: el dataset grande, mas epocas, y que los checkpoints se guarden en Drive.

In [ ]:
import yaml

with open("entrenamiento/config_validacion.yaml") as f:
    config = yaml.safe_load(f)

config.update({
    "data": f"{DATASET_DIR}/train.pkl",
    "valdata": f"{DATASET_DIR}/val.pkl",
    "load_chkpt": CHECKPOINT,
    "tokenizer": TOKENIZER,
    "epochs": EPOCHS,
    "batchsize": BATCHSIZE,
    "micro_batchsize": MICRO_BATCHSIZE,
    "lr": LR,
    "sample_freq": SAMPLE_FREQ,
    "save_freq": SAVE_FREQ,
    # OJO: este `debug` del yaml NO alcanza para desactivar wandb -- `parse_args`
    # (pix2tex/utils/utils.py) lo pisa con el default del CLI. Hay que pasar
    # --debug en la linea de comandos, y eso es lo que hace la celda 9.
    "debug": True,
    "model_path": f"{DRIVE_DIR}/checkpoints_real",
    "output_path": f"{DRIVE_DIR}/outputs_real",
    "name": "pix2tex_real",
    "test_samples": min(8, N_VAL),
})

with open("entrenamiento/config_real.yaml", "w") as f:
    yaml.safe_dump(config, f)

config

## 8.b Reanudar una corrida cortada (opcional)

Colab desconecta los entornos por inactividad y por limite de tiempo, y este
fine-tuning dura horas: es normal tener que continuarlo. Para reanudar, corre el
notebook entero de nuevo (el dataset se restaura del cache de Drive en segundos)
y pone `REANUDAR = True` aca antes de entrenar.

pix2tex no guarda el estado del optimizador, solo los pesos: la reanudacion
retoma los pesos del ultimo checkpoint y el contador de epocas, pero el
optimizador y el scheduler arrancan de cero. Para un fine-tuning con learning
rate bajo la diferencia es menor, pero conviene saberlo.

**Si es tu primera corrida, dejalo en `False` y segui de largo.**

In [ ]:
import glob
import os
import re

import yaml

REANUDAR = False  # True si Colab te desconecto y queres continuar desde el ultimo checkpoint

if REANUDAR:
    checkpoints = glob.glob(f"{DRIVE_DIR}/checkpoints_real/pix2tex_real/*.pth")
    assert checkpoints, (
        f"No hay checkpoints en {DRIVE_DIR}/checkpoints_real/pix2tex_real/. "
        "Si nunca completaste una corrida, entrena desde cero con REANUDAR = False."
    )
    ultimo = max(checkpoints, key=os.path.getmtime)
    # Los checkpoints se llaman pix2tex_real_e{epoca}_step{paso}.pth, con la epoca en base 1;
    # pix2tex corre `for e in range(args.epoch, args.epochs)`, asi que esa epoca en base 1
    # es exactamente el indice base 0 de la siguiente.
    epoca = int(re.search(r"_e(\d+)_step", os.path.basename(ultimo)).group(1))

    with open("entrenamiento/config_real.yaml") as f:
        config_reanudar = yaml.safe_load(f)
    config_reanudar["load_chkpt"] = ultimo
    config_reanudar["epoch"] = epoca
    with open("entrenamiento/config_real.yaml", "w") as f:
        yaml.safe_dump(config_reanudar, f)

    print(f"Reanudando desde {os.path.basename(ultimo)}: "
          f"epocas {epoca} a {config_reanudar['epochs']}.")
else:
    print("Corrida desde cero: fine-tuning sobre los pesos pre-entrenados de pix2tex.")

## 9. Correr el fine-tuning

Con estos parametros por defecto esto tarda bastante mas que la prueba de humo (horas, no minutos). Mantene la pestana de Colab abierta -- el entrenamiento en si cuenta como actividad, pero si cerras la pestana el runtime se puede desconectar igual.

`--debug` es obligatorio (igual que en la prueba de humo): `parse_args` sobreescribe el `debug: true` del yaml con el default `False` del CLI, asi que sin este flag `args.wandb` queda en `True` y el arranque revienta en `wandb.util.generate_id()`. Si queres tracking en wandb, saca el flag y corre `wandb login` antes.

In [ ]:
!python entrenamiento/entrenar.py --config entrenamiento/config_real.yaml --debug

## 10. Verificar los checkpoints guardados en Drive

In [ ]:
!ls -la {DRIVE_DIR}/checkpoints_real/pix2tex_real/

## 11. Proximo paso: medir, no asumir

Que el entrenamiento haya corrido y guardado checkpoints **no confirma que el modelo mejoro**. Antes de considerar este fine-tuning listo:

- Corre `pruebas/arnes_evaluacion.py` (o el criterio de calidad que uses para LaTeX) comparando el checkpoint nuevo contra los pesos pre-entrenados originales, sobre un conjunto de prueba que el modelo no haya visto en el fine-tuning. **`pruebas/` esta en `.gitignore`, asi que no viene en el clon de Colab**: baja el `.pth` de Drive y corre la evaluacion en local, o sube el arnes a la VM a mano.
- El dataset sintetico de este notebook no incluye ruido de escaneo/fotografia real ni el estilo tipografico real de tus PDF -- si la mejora no se sostiene sobre formulas reales, hace falta un dataset con ejemplos reales (o mas realistas) antes de dar el fine-tuning por bueno.
- Recorda la prioridad del proyecto: LaTeX primero, hasta que cumpla el criterio de calidad -- recien despues Markdown.